In [1]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [ ]:
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage
import multiprocessing as mp
import random
import os
import cv2
import pickle
import torch
import torch.nn as nn
import torchvision
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import Variable
import matplotlib.pyplot as plt
from tqdm import tqdm_notebook
from tqdm import tnrange
#Check GPU support, please do activate GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def plot_training_progress(losses, accuracies):
    """
    Plots training loss and accuracy over epochs.

    Args:
        losses (list): List of epoch losses.
        accuracies (list): List of epoch accuracies.
    """
    epochs = range(1, len(losses) + 1)

    plt.figure(figsize=(12, 5))

    # Create a plot for the loss
    ax1 = plt.gca()  # Get current axis
    ax1.plot(epochs, losses, label='Loss', color='blue', marker='o')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Loss', color='blue')
    ax1.tick_params(axis='y', labelcolor='blue')
    ax1.grid()

    # Create a second y-axis for accuracy
    ax2 = ax1.twinx()  # Create a twin axis sharing the same x-axis
    ax2.plot(epochs, accuracies, label='Accuracy', color='orange', marker='o')
    ax2.set_ylabel('Accuracy', color='orange')
    ax2.tick_params(axis='y', labelcolor='orange')

    # Combine legends
    lines, labels = ax1.get_legend_handles_labels()  # Get the loss labels
    lines2, labels2 = ax2.get_legend_handles_labels()  # Get the accuracy labels
    ax1.legend(lines + lines2, labels + labels2)  # Combine both legends

    plt.tight_layout()
    plt.show()
def extract_sample(n_way, n_support, n_query, datax, datay,K):
  """
  Picks random sample of size n_support+n_querry, for n_way classes
  Args:
      n_way (int): number of classes in a classification task
      n_support (int): number of labeled examples per class in the support set
      n_query (int): number of labeled examples per class in the query set
      datax (np.array): dataset of images
      datay (np.array): dataset of labels
  Returns:
      (dict) of:
        (torch.Tensor): sample of images. Size (n_way, n_support+n_query, (dim))
        (int): n_way
        (int): n_support
        (int): n_query
  """
  sample = []

  #extract samples from each claass
  for cls in K:
    datax_cls = datax[datay == cls]
    perm = np.random.permutation(datax_cls)
    sample_cls = perm[:(n_support+n_query)]
    sample.append(sample_cls)
  sample = np.array(sample)
  sample = torch.from_numpy(sample).float()
  sample = sample.permute(0,1,4,2,3)

  return({
      'images': sample,
      'n_way': n_way,
      'n_support': n_support,
      'n_query': n_query,
      'Classes':K
      })
class Flatten(nn.Module):
    def __init__(self):
        super(Flatten, self).__init__()

    def forward(self, x):
        return x.view(x.size(0), -1)

def load_protonet_conv(**kwargs):
    """
    Loads the prototypical network model
    Arg:
        x_dim (tuple): dimension of input image
        hid_dim (int): dimension of hidden layers in conv blocks
        z_dim (int): dimension of embedded image
    Returns:
        Model (Class ProtoNet)
    """
    x_dim = kwargs['x_dim']
    hid_dim = kwargs['hid_dim']
    z_dim = kwargs['z_dim']

    def conv_block(in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=2, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

    encoder = nn.Sequential(
        conv_block(x_dim[0], hid_dim),
        conv_block(hid_dim, hid_dim),
        conv_block(hid_dim, hid_dim),
        conv_block(hid_dim, z_dim),
        Flatten()
    )

    # Initialize weights using Kaiming Normal initialization
    for m in encoder.modules():
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):  # If there are linear layers
            nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)

    return ProtoNet(encoder)
class ProtoNet(nn.Module):
  def __init__(self, encoder):
    """
    Args:
        encoder : CNN encoding the images in sample
        n_way (int): number of classes in a classification task
        n_support (int): number of labeled examples per class in the support set
        n_query (int): number of labeled examples per class in the query set
    """
    super(ProtoNet, self).__init__()
    self.encoder = encoder.to(device)

  def set_forward_loss(self, sample):
    """
    Computes loss, accuracy and output for classification task
    Args:
        sample (torch.Tensor): shape (n_way, n_support+n_query, (dim))
    Returns:
        torch.Tensor: shape(2), loss, accuracy and y_hat
    """
    sample_images = sample['images'].to(device)
    n_way = sample['n_way']
    n_support = sample['n_support']
    n_query = sample['n_query']
    Classes=sample['Classes']

    x_support = sample_images[:, :n_support]
    x_query = sample_images[:, n_support:]

    #target indices are 0 ... n_way-1
    target_inds = torch.arange(0, n_way).view(n_way, 1, 1).expand(n_way, n_query, 1).long()
    target_inds = Variable(target_inds, requires_grad=False)
    target_inds = target_inds.to(device)

    #encode images of the support and the query set
    x = torch.cat([x_support.contiguous().view(n_way * n_support, *x_support.size()[2:]),
                   x_query.contiguous().view(n_way * n_query, *x_query.size()[2:])], 0)

    z = self.encoder.forward(x)
    z_dim = z.size(-1) #usually 64
    # Extract support and query samples
    z_samples = z[:n_way * n_support].view(n_way, n_support, z_dim)
    z_query = z[n_way * n_support:]

    # Ensure gradients are tracked
    z_samples.requires_grad_(True)
    z_query.requires_grad_(True)

    #----------------------------Making of prototypes---------------------------
    z_proto = z_samples.mean(dim=1)
    #-----------------calculationg distance from each prototypes----------------
    dists = euclidean_dist(z_query,z_proto)
    #--------------compute probabilities,loss and accuracy----------------------
    log_p_y = F.log_softmax(-dists, dim=1).view(n_way, n_query, -1)
    loss_val = -log_p_y.gather(2, target_inds).squeeze().view(-1).mean()
    _, y_hat = log_p_y.max(2)
    acc_val = torch.eq(y_hat, target_inds.squeeze()).float().mean()
    #-------------------------------------per class accurcay-------------
    # Compute per-class accuracy
    per_class_acc = torch.eq(y_hat, target_inds.squeeze()).float().mean(dim=1)  # Shape: (n_way,)

    return loss_val, {
        'loss': loss_val.item(),
        'acc': acc_val.item(),
        'y_hat': y_hat,
        'Classes':Classes,
        'per_class_acc':per_class_acc
        }
def euclidean_dist(x, y):
  """
  Computes euclidean distance btw x and y
  Args:
      x (torch.Tensor): shape (n, d). n usually n_way*n_query
      y (torch.Tensor): shape (m, d). m usually n_way
  Returns:
      torch.Tensor: shape(n, m). For each query, the distances to each centroid
  """
  n = x.size(0)
  m = y.size(0)
  d = x.size(1)
  assert d == y.size(1)

  x = x.unsqueeze(1).expand(n, m, d)
  y = y.unsqueeze(0).expand(n, m, d)

  return torch.pow(x - y, 2).sum(2)
def train(model, optimizer, train_x, train_y, n_way, n_support, n_query, max_epoch, epoch_size, CSR):
    """
    Trains the protonet
    Args:
        model
        optimizer
        train_x (np.array): images of training set
        train_y(np.array): labels of training set
        n_way (int): number of classes in a classification task
        n_support (int): number of labeled examples per class in the support set
        n_query (int): number of labeled examples per class in the query set
        max_epoch (int): max epochs to train on
        epoch_size (int): episodes per epoch
        CSR (float): class sampling rate for adaptive sampling
    """
    # Divide the learning rate by 2 at each epoch
    scheduler = optim.lr_scheduler.StepLR(optimizer, 1, gamma=0.5, last_epoch=-1)

    epoch = 0
    stop = False

    Tr_Classes = np.unique(train_y)
    epoch_losses = []
    epoch_accuracies = []

    # ✅ Initialize top_vulnerable_classes for the first epoch
    top_vulnerable_classes = list(np.random.choice(np.unique(train_y), n_way, replace=False))
    while epoch < max_epoch and not stop:
        running_loss = 0.0
        running_acc = 0.0

        # ---------------------- (ACS)Sampling strategy for each episode -------------------------
        ours_episode_count = int(epoch_size * CSR)
        sampling_strategy = [True] * ours_episode_count + [False] * (epoch_size - ours_episode_count)
        random.shuffle(sampling_strategy)

        # ---------------------- Accuracy dictionary for each epoch -------------------------
        accuracy_dict = {class_name: [] for class_name in Tr_Classes}

        for episode in tnrange(epoch_size, desc="Epoch {:d} train".format(epoch + 1)):
            # Sampling class set K
            if sampling_strategy[episode] or episode == 0:
                K = np.random.choice(np.unique(train_y), n_way, replace=False)
            else:
                K = top_vulnerable_classes

            sample = extract_sample(n_way, n_support, n_query, train_x, train_y, K)
            optimizer.zero_grad()
            loss, output = model.set_forward_loss(sample)
            running_loss += output['loss']
            running_acc += output['acc']
            loss.backward()
            optimizer.step()

            # ------------------ Collect class-wise accuracy from this episode ------------------
            Out_Classes = output['Classes']
            Out_per_class_acc = output['per_class_acc']
            for class_name, class_acc in zip(Out_Classes, Out_per_class_acc):
                accuracy_dict[class_name].append(class_acc)

        # ------------------- Compute vulnerability after the epoch -------------------
        class_probabilities_float = {cls: [p.item() for p in probs] for cls, probs in accuracy_dict.items()}
        mean_confidences = {cls: np.mean(probs) for cls, probs in class_probabilities_float.items() if len(probs) > 0}
        sorted_classes = sorted(mean_confidences.items(), key=lambda x: x[1])  # Ascending
        top_vulnerable_classes = [cls for cls, _ in sorted_classes[:n_way]]

        # ------------------- Logging and learning rate scheduling -------------------
        epoch_loss = running_loss / epoch_size
        epoch_acc = running_acc / epoch_size
        print('Epoch {:d} -- Loss: {:.4f} Acc: {:.4f}'.format(epoch + 1, epoch_loss, epoch_acc))

        epoch_losses.append(epoch_loss)
        epoch_accuracies.append(epoch_acc)

        epoch += 1
        scheduler.step()


    # Call the plotting function after training
    #plot_training_progress(epoch_losses, epoch_accuracies)

def test(model, test_x, test_y, n_way, n_support, n_query, test_episode):
    """
    Tests the protonet
    Args:
        model: trained model
        test_x (np.array): images of testing set
        test_y (np.array): labels of testing set
        n_way (int): number of classes in a classification task
        n_support (int): number of labeled examples per class in the support set
        n_query (int): number of labeled examples per class in the query set
        test_episode (int): number of episodes to test on
    """
    model.eval()

    loss_list = []
    acc_list = []

    running_loss = 0.0
    running_acc = 0.0
    for episode in range(test_episode):
        K = np.random.choice(np.unique(test_y), n_way, replace=False)
        sample = extract_sample(n_way, n_support, n_query, test_x, test_y,K)
        loss, output = model.set_forward_loss(sample)

        running_loss += output['loss']
        running_acc += output['acc']

        # Append the current loss and accuracy for later plotting
        loss_list.append(output['loss'])
        acc_list.append(output['acc'])

    avg_loss = running_loss / test_episode
    avg_acc = running_acc / test_episode
    print('Test results -- Loss: {:.4f} Acc: {:.4f}'.format(avg_loss, avg_acc))
    mean_accuracy,margin_of_error=Calc_Conf_Acc(acc_list)
    return mean_accuracy,margin_of_error
def Calc_Conf_Acc(Calc_Test_Conf):
  flattened_Calc_Test_Conf = Calc_Test_Conf

  mean_accuracy = np.mean(flattened_Calc_Test_Conf)

  std_dev = np.std(flattened_Calc_Test_Conf)

  # Calculate standard error of the mean
  std_error = stats.sem(flattened_Calc_Test_Conf)

  # Calculate margin of error for 95% confidence interval
  margin_of_error = stats.t.ppf(0.975, len(flattened_Calc_Test_Conf)-1) * std_error

  # Calculate lower and upper bounds of the confidence interval
  lower_bound = mean_accuracy - margin_of_error
  upper_bound = mean_accuracy + margin_of_error
  return mean_accuracy,margin_of_error


#-----------Main function-----------------------------------------------------

n_way = 2
n_support = 1
n_query = 5
epochs_to_run = [20,22,25,30,35,40]
epoch_size = 20
test_episode = 60
Size=84
#-------------------------set seeeds-----------------------------
np_seed = 0 # Set the seed for reproducibility
torch_seed = 0
CSR=0.5

# Fix all seeds
np.random.seed(np_seed)
torch.manual_seed(torch_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(torch_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Load pickle files
PATH = '/content/gdrive/MyDrive/Refined-Pickle-Files/Refined-Pickle-Files/Pickle-'+str(Size)+'_'+str(Size)+'/'
Data='Aug_Skin'

with open(PATH + Data + "_trainx.pkl", "rb") as f:
    trainx = pickle.load(f)
with open(PATH + Data + "_trainy.pkl", "rb") as f:
    trainy = pickle.load(f)
with open(PATH + Data + "_testx.pkl", "rb") as f:
    testx = pickle.load(f)
with open(PATH + Data + "_testy.pkl", "rb") as f:
    testy = pickle.load(f)

# Function to track the highest accuracy
def get_accuracy(model, testx, testy, n_way, n_support, n_query, test_episode):
    # Assuming this function is available in your framework
    return test(model, testx, testy, n_way, n_support, n_query, test_episode)

best_accuracy = 0
best_epoch = 0

# Training and testing for specified epochs
for epoch in epochs_to_run:
    # Initialize model and optimizer
    model = load_protonet_conv(x_dim=(3, Size, Size), hid_dim=64, z_dim=64)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    print(f"-----------------------Training for {epoch} epochs-----------------------")
    train(model, optimizer, trainx, trainy, n_way, n_support, n_query, epoch, epoch_size,CSR)

    print(f"-----------------------Testing after {epoch} epochs-----------------------")
    mean_accuracy,margin_of_error = get_accuracy(model, testx, testy, n_way, n_support, n_query, test_episode)

    print(f"Accuracy after {epoch} epochs: {mean_accuracy}")

    if mean_accuracy > best_accuracy:
        best_accuracy = mean_accuracy
        best_epoch = epoch
        best_mar=margin_of_error

print(f"\nHighest accuracy: {best_accuracy} and error margin: {best_mar} achieved at epoch: {best_epoch}")


-----------------------Training for 20 epochs-----------------------


/tmp/ipython-input-1751166042.py:264: TqdmDeprecationWarning: Please use `tqdm.notebook.trange` instead of `tqdm.tnrange`
  for episode in tnrange(epoch_size, desc="Epoch {:d} train".format(epoch + 1)):


Epoch 1 train:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 1 -- Loss: 201.5803 Acc: 0.6300


Epoch 2 train:   0%|          | 0/20 [00:00<?, ?it/s]

...

Epoch 20 train:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 20 -- Loss: 27.9463 Acc: 0.6400


Epoch 21 train:   0%|          | 0/20 [00:00<?, ?it/s]

Epoch 21 -- Loss: 44.1091 Acc: 0.6100


Epoch 22 train:   0%|          | 0/20 [00:00<?, ?it/s]